read initial zones file and bus stops

In [ ]:
import geopandas as gpd

# transport zones
tz_gdf = gpd.read_file("../initial_transport_zones.SHP")

# bus stops
osm_stops = gpd.read_file("../bus_stops.geojson")

define crs

In [2]:
utm_crs = tz_gdf.estimate_utm_crs()
crs_4326 = 4326

read file with routes info

In [ ]:
import json

# routes 
with open('../routes.json', 'r', encoding='utf-8') as f:
    route_path_dict_stop_names = json.load(f)


from connectpt.preprocess import create_stops_gdf_from_routes

stops_routes_gdf = create_stops_gdf_from_routes(route_path_dict_stop_names)

define boundary of total territory

get drive and bus graphs of defined area

In [4]:
from connectpt.preprocess import get_boundary_gdf
from connectpt.preprocess import get_drive_graph_iduedu, get_bus_graph_iduedu

territory = get_boundary_gdf(tz_gdf)
territory_utm = territory.to_crs(utm_crs).geometry.iloc[0]
territory_4326 = territory.to_crs(crs_4326).geometry.iloc[0]
G_drive, G_drive_edges, G_drive_nodes = get_drive_graph_iduedu(territory_4326)
G_pt, G_pt_edges, G_pt_nodes = get_bus_graph_iduedu(territory_4326)

2026-07-09 17:20:01.263 | INFO     | Downloading drive network via Overpass ...
2026-07-09 17:20:01.401 | INFO     | Downloading network via Overpass done!
2026-07-09 17:20:04.834 | WARNING  | Removing 190 nodes from 78 smaller strongly connected components. These are subgraphs where nodes are internally reachable but isolated from the rest. Retaining only the largest strongly connected component (7576 nodes).
2026-07-09 17:20:05.415 | INFO     | Downloading routes via Overpass with types bus ...
2026-07-09 17:20:05.596 | INFO     | Downloading routes via Overpass with types bus done!
Parsing ground PT routes: 100%|██████████| 206/206 [00:06<00:00, 30.52it/s]


 download water by defined area (optional)

In [5]:
import osmnx as ox

bc_tags = {
    'roads': {
      "highway": ["construction","crossing","living_street","motorway","motorway_link","motorway_junction","pedestrian","primary","primary_link","raceway","residential","road","secondary","secondary_link","services","tertiary","tertiary_link","track","trunk","trunk_link","turning_circle","turning_loop","unclassified",],
      "service": ["living_street", "emergency_access"]
    },
    'railways': {
      "railway": "rail"
    },
    'water': {
      'riverbank':True,
      'reservoir':True,
      'basin':True,
      'dock':True,
      'canal':True,
      'pond':True,
      'natural':['water','bay'],
      'waterway':['river','canal','ditch'],
      'landuse':'basin',
      'water': 'lake'
    }
}


water = ox.features_from_polygon(territory_4326, bc_tags['water'])


preprocessing stops names 

In [6]:
from connectpt.preprocess import preprocess_stops_names_gdf

osm_stops_preprocessed = preprocess_stops_names_gdf(osm_stops)

clusterization

In [7]:
from connectpt.preprocess import cluster_stops

clusters_gdf, all_points_gdf = cluster_stops(
    osm_stops_preprocessed.to_crs(utm_crs), 
    distance_threshold=700, 
    jaccard_threshold=0.65
)

build new zones based on clusters

In [8]:
from connectpt.preprocess import build_new_zones

voronoi_gdf = build_new_zones(
    all_points_gdf,
    utm_crs=utm_crs,
    territory= territory,
    clusters_gdf=clusters_gdf,
    water_gdf=water,
    remove_water=True,
    clip_by_territory=True,
    merge_to_zones=True
)

Preparing data...
Building voronoi diagram...
Removing water objects...
Clipping territory by defined are...
Comparing polygons and bus stops...
Merging voronoi polygons to zones...
Done!


make block graph

In [9]:
from connectpt.preprocess import make_block_graph


block_graph = make_block_graph(
    voronoi_gdf,
    road_edges_gdf=G_drive_edges,
    territory=territory,
    utm_crs=utm_crs,
)

Preparing data for block graph making...
Adding nodes to block graph...
Finding touching neighbors polygons...
Finding road connections...


Checking roads for connection: 100%|██████████| 18221/18221 [00:19<00:00, 923.32road/s] 


Uniting all connections..
Adding edges to block graph...
Done!


In [10]:
from connectpt.preprocess import pt_graph_project_by_stops_name


block_graph_with_routes, block_project_dict, pt_graph = pt_graph_project_by_stops_name(
    block_graph=block_graph,
    dict_routes=route_path_dict_stop_names,
    polygons_gdf=voronoi_gdf,
    tr_edges_gdf=G_pt_edges,
    drive_edges_gdf=G_drive_edges,
    territory=territory, 
    utm_crs=utm_crs)

Preparing data for pt_graph...
Comparing stops and zones...
Projecting pt routes and building pt graph...
Done!


get nodes and edges geodataframes from block graph

In [11]:
from connectpt.preprocess import block_graph_to_gdfs


block_graph_nodes_gdf, block_graph_edges_gdf = block_graph_to_gdfs(block_graph_with_routes)

get road segments connecting new zones

In [ ]:
from connectpt.preprocess import get_road_segments_for_block_graph_edges


road_segments_gdf = get_road_segments_for_block_graph_edges(block_graph_edges_gdf)

create geodataframes with edges as routes parts 

one edge - one route segment between zones

In [13]:
from connectpt.preprocess import create_routes_geodataframe


routes_edges_gdf = create_routes_geodataframe(block_graph_with_routes, block_project_dict)

read initial od matrix

In [ ]:
import pandas as pd
import numpy as np

od_df_readed_2 = pd.read_excel('../matrix.xlsx', sheet_name=1)
od_df_readed_3 = pd.read_excel('../matrix.xlsx', sheet_name=2)
clean_df_2 = od_df_readed_2.iloc[2:, 3:].reset_index(drop=True)
clean_df_3 = od_df_readed_3.iloc[2:, 3:].reset_index(drop=True)
# clean_df = clean_df.reset_index(drop=True)

od_matrix_initial_2 = clean_df_2.values.tolist()
od_matrix_initial_3 = clean_df_3.values.tolist()
od_matrix_initial_2 = np.array(od_matrix_initial_2, dtype=np.float64)
od_matrix_initial_3 = np.array(od_matrix_initial_3, dtype=np.float64)

od_matrix_initial = od_matrix_initial_2 + od_matrix_initial_3
# od_matrix_initial = od_matrix_initial_3

In [15]:
from connectpt.preprocess import reform_od_matrix


new_od_matrix = reform_od_matrix(od_matrix_initial,
                 tz_gdf,
                 voronoi_gdf,
                 utm_crs,
                 water,
                 remove_water=True)

Checking and preparing data...
Calculatind intersections for zones...


Computing new matrix: 100%|██████████| 237/237 [00:01<00:00, 148.21it/s]

Done!


save files for route optimization

In [26]:
from connectpt.preprocess import save_files

save_files(
    output_folder="data_test",
    city_name="City",
    block_graph=block_graph_with_routes,
    od_matrix=new_od_matrix,
    block_project_dict=block_project_dict,
    save_coords=True,
    save_demand=True,
    save_travel_times=True,
    save_routes=True
)


data_test//CityCoords.txt saved
data_test//CityDemand.txt saved
data_test//CityTravelTimes.txt saved
data_test//CityRoutes.pkl saved
